In [1]:
import sys 
from pathlib import Path
sys.path.append(str(Path().resolve().parents[1]))

BASE_DIR = Path().resolve()
PROJECT_ROOT = BASE_DIR.parent   #to go un up the the root
DATA_DIR = PROJECT_ROOT / "data" 



from python.dataset_creation import (unique_subjects, all_subjects, 
                                    X_glob,X_seq, y                                        
                                    )



from utilities.model import  CNNMLPModel, CNNMLPModelSmall



import torch
import numpy as np

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

subject=s1, exercise=e1, label=correct, unit=u2, axis=z, valleys=11
subject=s1, exercise=e1, label=fast, unit=u2, axis=z, valleys=11
subject=s1, exercise=e1, label=low_amplitude, unit=u2, axis=z, valleys=11
subject=s1, exercise=e2, label=correct, unit=u4, axis=y, valleys=11
subject=s1, exercise=e2, label=fast, unit=u4, axis=y, valleys=11
subject=s1, exercise=e2, label=low_amplitude, unit=u4, axis=y, valleys=11
subject=s1, exercise=e3, label=correct, unit=u2, axis=z, valleys=10
subject=s1, exercise=e3, label=fast, unit=u2, axis=z, valleys=11
subject=s1, exercise=e3, label=low_amplitude, unit=u2, axis=z, valleys=11
subject=s1, exercise=e4, label=correct, unit=u2, axis=y, valleys=11
subject=s1, exercise=e4, label=fast, unit=u2, axis=y, valleys=11
subject=s1, exercise=e4, label=low_amplitude, unit=u2, axis=y, valleys=11
subject=s1, exercise=e5, label=correct, unit=u2, axis=x, valleys=11
subject=s1, exercise=e5, label=fast, unit=u2, axis=x, valleys=11
subject=s1, exercise=e5, label=low_ampl

In [2]:
print(type(all_subjects))
print(np.array(all_subjects).shape)
print(all_subjects[:10] if hasattr(all_subjects, "__len__") else all_subjects)

<class 'numpy.ndarray'>
(1193,)
['s1' 's1' 's1' 's1' 's1' 's1' 's1' 's1' 's1' 's1']


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



# cross validation

In [4]:

def normalize_seq_per_subject(X, subjects):
    X_norm = torch.empty_like(X)

    for s in set(subjects):
        idx = [i for i, subj in enumerate(subjects) if subj == s]

        # If X shape is [samples, time, channels]
        mean = X[idx].mean(dim=(0, 1), keepdim=True)
        std  = X[idx].std(dim=(0, 1), keepdim=True) + 1e-6

        X_norm[idx] = (X[idx] - mean) / std

    return X_norm

fold_results = []

for val_subject in unique_subjects:
    print(f"\n========== Fold: leaving out {val_subject} ==========")

    train_indices = [i for i, s in enumerate(all_subjects) if s != val_subject]
    val_indices = [i for i, s in enumerate(all_subjects) if s == val_subject]

    print(train_indices)

    X_seq_train = X_seq[train_indices]
    X_seq_val   = X_seq[val_indices]

    train_subjects = [all_subjects[i] for i in train_indices]
    val_subjects   = [all_subjects[i] for i in val_indices]

    '''X_seq_train = normalize_seq_per_subject(X_seq_train, train_subjects)
    X_seq_val   = normalize_seq_per_subject(X_seq_val, val_subjects)'''
    

    X_glob_train = X_glob[train_indices]
    X_glob_val   = X_glob[val_indices]

    y_train = y[train_indices]
    y_val   = y[val_indices]

    '''  train_mean = X_train.mean(dim=(0,2), keepdim=True)
    train_std = X_train.std(dim=(0,2), keepdim=True) + 1e-6

    X_train = (X_train - train_mean) / train_std
    X_val = (X_val - train_mean) / train_std'''




    
    train_dataset = TensorDataset(
    X_seq_train,
    X_glob_train,
    y_train
)

    val_dataset = TensorDataset(
    X_seq_val,
    X_glob_val,
    y_val
)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    # fresh model for each fold
    model = CNNMLPModel(input_dim_seq=15,input_dim_global=9, num_classes=3).to(device)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-3
)

    best_val_loss = float("inf")
    best_preds = None
    best_labels = None

    for epoch in range(40):
        # TRAIN
        model.train()
        running_loss = 0.0

        for x_seq_b, x_glob_b, y_b in train_loader:
            x_seq_b = x_seq_b.to(device)
            x_glob_b = x_glob_b.to(device)
            y_b = y_b.to(device)

            optimizer.zero_grad()

            outputs = model(x_seq_b, x_glob_b)
            loss = criterion(outputs, y_b)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x_seq_b.size(0)

        train_loss = running_loss / len(train_loader.dataset)

        # VALIDATION
        model.eval()
        val_running_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for x_seq_b, x_glob_b, y_b in val_loader:
                x_seq_b = x_seq_b.to(device)
                x_glob_b = x_glob_b.to(device)
                y_b = y_b.to(device)

                outputs = model(x_seq_b, x_glob_b)
                loss = criterion(outputs, y_b)

                val_running_loss += loss.item() * x_seq_b.size(0)

                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_b.cpu().numpy())

        val_loss = val_running_loss / len(val_loader.dataset)
        val_acc = accuracy_score(all_labels, all_preds)

        print(
            f"Fold {val_subject} | Epoch {epoch+1} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        # keep best epoch for this fold
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_preds = np.array(all_preds)
            best_labels = np.array(all_labels)

    # fold-level metrics
    fold_acc = accuracy_score(best_labels, best_preds)
    fold_f1 = f1_score(best_labels, best_preds, average="macro")
    fold_cm = confusion_matrix(best_labels, best_preds)

    fold_results.append({
        "subject": val_subject,
        "val_loss": best_val_loss,
        "accuracy": fold_acc,
        "macro_f1": fold_f1,
        "confusion_matrix": fold_cm
    })

    


========== Fold: leaving out s1 ==========
[238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 

In [5]:
for r in fold_results:
    print(f"Subject: {r['subject']}")
    print(f"Val Loss: {r['val_loss']:.4f}")
    print(f"Accuracy: {r['accuracy']:.4f}")
    print(f"Macro F1: {r['macro_f1']:.4f}")
    print("Confusion Matrix:")
    print(r["confusion_matrix"])
    print("-" * 40)

Subject: s1
Val Loss: 0.7354
Accuracy: 0.6513
Macro F1: 0.5958
Confusion Matrix:
[[14 52 13]
 [ 0 69 10]
 [ 0  8 72]]
----------------------------------------
Subject: s2
Val Loss: 0.3041
Accuracy: 0.8875
Macro F1: 0.8864
Confusion Matrix:
[[61 19  0]
 [ 1 72  7]
 [ 0  0 80]]
----------------------------------------
Subject: s3
Val Loss: 0.4296
Accuracy: 0.8445
Macro F1: 0.8430
Confusion Matrix:
[[62 17  0]
 [13 61  5]
 [ 2  0 78]]
----------------------------------------
Subject: s4
Val Loss: 0.5077
Accuracy: 0.7689
Macro F1: 0.7589
Confusion Matrix:
[[35 44  1]
 [ 3 75  1]
 [ 0  6 73]]
----------------------------------------
Subject: s5
Val Loss: 0.3834
Accuracy: 0.8368
Macro F1: 0.8387
Confusion Matrix:
[[62  3 15]
 [ 1 58 20]
 [ 0  0 80]]
----------------------------------------


In [6]:
for s in np.unique(all_subjects):
    mask = np.array(all_subjects) == s
    print(s, np.bincount(y[mask]))

s1 [79 79 80]
s2 [80 80 80]
s3 [79 79 80]
s4 [80 79 79]
s5 [80 79 80]


# model training 


In [7]:
val_subject = 's2'   # change this to the patient you want to leave out

train_indices = [
    i for i, s in enumerate(all_subjects)
    if s != val_subject
]

val_indices = [
    i for i, s in enumerate(all_subjects)
    if s == val_subject
]

X_seq_train = X_seq[train_indices]
X_seq_val = X_seq[val_indices]

X_glob_train = X_glob[train_indices]
X_glob_val = X_glob[val_indices]

y_train = y[train_indices]
y_val = y[val_indices]

print("Training samples:", len(train_indices))
print("Validation samples:", len(val_indices))
print("Left-out subject:", val_subject)

Training samples: 953
Validation samples: 240
Left-out subject: s2


In [8]:
import copy
best_model_state = None

train_dataset = TensorDataset(
    X_seq_train,
    X_glob_train,
    y_train
)

val_dataset = TensorDataset(
    X_seq_val,
    X_glob_val,
    y_val
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# fresh model for each fold
model = CNNMLPModel(
    input_dim_seq=15,
    input_dim_global=9,
    num_classes=3
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

best_val_loss = float("inf")
best_preds = None
best_labels = None

for epoch in range(20):
    # TRAIN
    model.train()
    running_loss = 0.0

    for x_seq_b, x_glob_b, y_b in train_loader:
        x_seq_b = x_seq_b.to(device)
        x_glob_b = x_glob_b.to(device)
        y_b = y_b.to(device)

        optimizer.zero_grad()

        outputs = model(x_seq_b, x_glob_b)
        loss = criterion(outputs, y_b)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x_seq_b.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    # VALIDATION
    model.eval()
    val_running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x_seq_b, x_glob_b, y_b in val_loader:
            x_seq_b = x_seq_b.to(device)
            x_glob_b = x_glob_b.to(device)
            y_b = y_b.to(device)

            outputs = model(x_seq_b, x_glob_b)
            loss = criterion(outputs, y_b)

            val_running_loss += loss.item() * x_seq_b.size(0)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_b.cpu().numpy())

    val_loss = val_running_loss / len(val_loader.dataset)
    val_acc = accuracy_score(all_labels, all_preds)

    print(
        f"Fold {val_subject} | Epoch {epoch + 1} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    # keep best epoch for this fold
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_preds = np.array(all_preds)
        best_labels = np.array(all_labels)
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch + 1

# fold-level metrics
acc = accuracy_score(best_labels, best_preds)
f1 = f1_score(best_labels, best_preds, average="macro")
cm = confusion_matrix(best_labels, best_preds)

results = {}

results[val_subject] = {
    "subject": val_subject,
    "best_epoch": best_epoch,
    "val_loss": best_val_loss,
    "accuracy": fold_acc,
    "macro_f1": fold_f1,
    "confusion_matrix": fold_cm,
    "model_state_dict": best_model_state,
    "predictions": best_preds,
    "labels": best_labels
}

Fold s2 | Epoch 1 | Train Loss: 1.0404 | Val Loss: 0.9977 | Val Acc: 0.5500
Fold s2 | Epoch 2 | Train Loss: 0.9236 | Val Loss: 0.9385 | Val Acc: 0.4750
Fold s2 | Epoch 3 | Train Loss: 0.8490 | Val Loss: 0.9922 | Val Acc: 0.3750
Fold s2 | Epoch 4 | Train Loss: 0.7746 | Val Loss: 0.6890 | Val Acc: 0.7833
Fold s2 | Epoch 5 | Train Loss: 0.7230 | Val Loss: 0.6254 | Val Acc: 0.7208
Fold s2 | Epoch 6 | Train Loss: 0.6811 | Val Loss: 0.8617 | Val Acc: 0.5375
Fold s2 | Epoch 7 | Train Loss: 0.6520 | Val Loss: 0.9235 | Val Acc: 0.4708
Fold s2 | Epoch 8 | Train Loss: 0.6134 | Val Loss: 0.8040 | Val Acc: 0.6458
Fold s2 | Epoch 9 | Train Loss: 0.5929 | Val Loss: 0.8392 | Val Acc: 0.5917
Fold s2 | Epoch 10 | Train Loss: 0.5539 | Val Loss: 0.7483 | Val Acc: 0.7000
Fold s2 | Epoch 11 | Train Loss: 0.5408 | Val Loss: 0.4918 | Val Acc: 0.8000
Fold s2 | Epoch 12 | Train Loss: 0.5180 | Val Loss: 0.5255 | Val Acc: 0.7500
Fold s2 | Epoch 13 | Train Loss: 0.5092 | Val Loss: 0.4606 | Val Acc: 0.7917
Fold s2 

In [9]:
torch.save(results, "fold_results.pth")